# Single Gene Deletion study of extended iM1515 models

- Simulate SGKO for extended genome-scale metabolic models related to iML1515
- Models have been adapted to Keio collection strains and generated using Jupyter notebooks (iML1515_Keio), via f2xba Python framework.
- Optimization performed using gurobipy interface, which if significantly faster than CobraPy
- Keio collection strains are based on BW25311 with genetic background ∆araBAD ∆rhaBAD ∆lacZ
    - b0063\araB, b0062\araA, b0061\araD, b3904\rhaB, b3903\rhaA, b3902\rhaD, b0344\lacZ
- Simulations were performed on Python 3.13, using f2xba 1.11.0, and Gurobi 13.0.3.

Peter Schubert, Heinrich-Heine University Duesseldorf, Institute for Computational Cell Biology (M. Lercher), Nov 11, 2026

## Initial setup

In [1]:
# Required imports and model names
import os
import pandas as pd

from f2xba import FbaOptimization, EcmOptimization, RbaOptimization

pd.set_option('io.excel.xlsx.reader', 'calamine')

# directories and model names
sbml_dir = 'SBML_models/ecoli'
optim_data_dir = 'data_optim/ecoli'
plots_dir = 'plots/ecoli'
results_dir = 'results/ecoli'

# models
fba_model = 'iML1515_updated'
gecko_turnup_model = 'iML1515_predicted_GECKO'
gecko_model = 'iML1515_predicted_fit_GECKO'
rba_model = 'iML1515_RBA'
tfa_model = 'iML1515_TFA'
tgecko_model = 'iML1515_TGECKO'
trba_model = 'iML1515_TRBA'

medium_fname = 'iML1515_LB_medium.xlsx'        # Media composition for simulation
print(f'working directory: {os.getcwd()}')

working directory: /Users/peter/Documents/CCB/work/my_publications/Deya/notebooks


In [2]:
models = {fba_model:          {'type': 'FBA', 'td': False, 'dir': sbml_dir},
          gecko_turnup_model: {'type': 'ECM', 'td': False, 'dir': sbml_dir},
          gecko_model:        {'type': 'ECM', 'td': False, 'dir': sbml_dir},
          rba_model:          {'type': 'RBA', 'td': False, 'dir': sbml_dir},
          tfa_model:          {'type': 'FBA', 'td': True,  'dir': sbml_dir},
          tgecko_model:       {'type': 'ECM', 'td': True,  'dir': sbml_dir},
          trba_model:         {'type': 'RBA', 'td': True,  'dir': sbml_dir},}
remap_solution_names = {fba_model: 'iML1515_FBA', gecko_turnup_model: 'iML1515_TurNuP_GECKO', gecko_model: 'iML1515_GECKO'}

In [3]:
# Load experimental conditions

# Load LB medium used of SGKO analysis
fname = os.path.join(optim_data_dir, medium_fname)
with pd.ExcelFile(fname) as xlsx:
    lb_medium = pd.read_excel(xlsx, sheet_name='lb_medium', index_col=0)['max_uptake'].to_dict()
print(f'lb medium ({len(lb_medium)} nutrients) loaded from {fname}')

# Metabolite concentrations to be fixed with a margin (Buckstein, 2007)
nt_concs_µmol_per_l = {'atp_c': 3560, 'adp_c': 116, 'datp_c': 181, 'ctp_c': 325, 'dctp_c': 184, 'gtp_c': 1660, 
                       'gdp_c': 203, 'dgtp_c': 92, 'utp_c': 667, 'udp_c': 54, 'dttp_c': 256, 'ppgpp_c': 113, 'accoa_c': 1390}
factor = 2.0
metabolite_molar_concs = {sid: (µmol_per_l * 1e-6/factor, µmol_per_l*1e-6*factor) 
                          for sid, µmol_per_l in  nt_concs_µmol_per_l.items()}

lb medium (43 nutrients) loaded from ../../iML1515_Keio/notebooks/data_optim/iML1515_LB_medium.xlsx
2349 records of proteomics loaded from ../../iML1515_Keio/notebooks/data_refs/Ecoli_Schmidt_proteomics.xlsx
2234 records with confidence level above 43.0


## SGKO for extended models
- SGKO optimization is performed for selected genome scale metabolic models using f2xba modeling framework
    - optimization for genes not used in the wild type solutions are considered to have not growth impact
- SGKO results are exported. 
- MOMA, linear MOMA, ROOM and linear ROOM methods can be applied to FBA and GECKO models (Note: simulating ROOM takes a long time)

In [4]:
# FBA and GECKO type SGKO simulations 
# genetic background of BW25311 wrt MG1655 ∆araBAD ∆rhaBAD ∆lacZ
# b0063\araB, b0062\araA, b0061\araD, b3904\rhaB, b3903\rhaA, b3902\rhaD, b0344\lacZ

# [fba_model, gecko_turnup_model, gecko_model, tfa_model, tgecko_model, rba_model, trba_model]
model_names = [fba_model, gecko_turnup_model, gecko_model, tfa_model, tgecko_model, rba_model, trba_model]
special_methods =  ['linear moma', 'moma', 'linear room', 'room']  # 'linear moma', 'moma', 'linear room', 'room'
analysis_names= {'moma': 'MOMA', 'linear moma': 'lMOMA', 'room': 'ROOM', 'linear room': 'lROOM'}

strain_background = {'b0063', 'b0062', 'b0061', 'b3904', 'b3903', 'b3902', 'b0344' }

for model_name in model_names:
    meta_data = models[model_name]

    # set medium
    if meta_data['type'] == 'FBA':
        xo = FbaOptimization(os.path.join(meta_data['dir'], f'{model_name}.xml'))
        xo.medium = lb_medium
    elif meta_data['type'] == 'ECM':
        xo = EcmOptimization(os.path.join(meta_data['dir'], f'{model_name}.xml'))
        xo.medium = {rid: 1000.0 for rid in lb_medium}
    elif meta_data['type'] == 'RBA':
        xo = RbaOptimization(os.path.join(meta_data['dir'], f'{model_name}.xml'))
        sigma = xo.avg_enz_saturation
        rel_mmol_per_l =  (sigma / (1.0-min(sigma, .99))) * xo.importer_km
        xo.medium = {re.sub('EX_', '', ex_ridx): rel_mmol_per_l for ex_ridx in lb_medium}
    
    # fix metabolite concentrations for TD models
    if meta_data['td'] is True:
        orig_concs = xo.set_tfa_metab_concentrations(metabolite_molar_concs)
        print(f'{len(orig_concs)} metabolite concentrations constrained')
    
    # set strain background
    orig_bounds = {}
    if strain_background is not None:
        orig_bounds.update(xo.gene_knock_outs(strain_background))
        print(f'Genetic background, genes deleted: {strain_background}')
    
    if meta_data['type'] == 'RBA':
        df_sgko = xo.single_gene_deletion(gr_min=0.01, gr_min_alt=0.02, gr_max=1.5, bisection_tol=.5e-3)
    else:
        # baseline model sgko analysis
        df_sgko = xo.single_gene_deletion()
        
        # special methods MOMA and ROOM for FBA and GECKO
        if model_name in {fba_model, gecko_model}:
            for special_method in special_methods:
                print(f'  {special_method} on {model_name}')
                if 'moma' in special_method:
                    df_sp_sgko = xo.single_gene_deletion(method=special_method)
                elif 'room' in special_method:
                    df_sp_sgko = xo.single_gene_deletion(method=special_method, delta=0.03, epsilon=1e-3, time_limit=30.0)
                remapped_name = remap_solution_names.get(model_name, model_name)
                sp_fname = os.path.join(results_dir, f'sgko_{remapped_name}_{analysis_names[special_method]}.xlsx')        
                with pd.ExcelWriter(sp_fname) as writer:
                    df_sp_sgko.to_excel(writer, sheet_name='sgko')
                    
    # export SGKO results (excluding special methods)
    remapped_name = remap_solution_names.get(model_name, model_name)
    fname = os.path.join(results_dir, f'sgko_{remapped_name}.xlsx')
    with pd.ExcelWriter(fname) as writer:
        df_sgko.to_excel(writer, sheet_name='sgko')

SBML model loaded by sbmlxdf: ../../iML1515_Keio/notebooks/SBML_models/iML1515_updated.xml (Tue Nov  3 11:28:33 2026)
Set parameter Username
Set parameter LicenseID to value 2835868
Academic license - for non-commercial use only - expires 2027-06-16
LP Model of iML1515
2713 variables, 1878 constraints, 10560 non-zero matrix coefficients
Genetic background, genes deleted: {'b3904', 'b3902', 'b0062', 'b0061', 'b3903', 'b0063', 'b0344'}
448 genes potentially active given flux distribution.


100%|██████████████████████████████████████| 1518/1518 [00:28<00:00, 53.14it/s]


  linear moma on iML1515_updated
445 genes potentially active given flux distribution.


100%|██████████████████████████████████████| 1518/1518 [01:28<00:00, 17.15it/s]


  moma on iML1515_updated
445 genes potentially active given flux distribution.


100%|██████████████████████████████████████| 1518/1518 [01:35<00:00, 15.94it/s]


... (20 outputs left out)

SBML model loaded by sbmlxdf: ../../iML1515_Keio/notebooks/SBML_models/iML1515_RBA.xml (Tue Nov  3 18:06:04 2026)
LP Model of iML1515_RBA
9677 variables, 8916 constraints, 123047 non-zero matrix coefficients
3750 enzymes, 6 process machines
RBA enzyme efficiency constraints configured (C_EF_xxx, C_ER_xxx) ≤ 0
Genetic background, genes deleted: {'b3904', 'b3902', 'b0062', 'b0061', 'b3903', 'b0063', 'b0344'}
411 genes potentially active given flux distribution.


100%|██████████████████████████████████████| 1613/1613 [13:25<00:00,  2.00it/s]


SBML model loaded by sbmlxdf: ../../iML1515_Keio/notebooks/SBML_models/iML1515_TRBA.xml (Tue Nov  3 18:14:31 2026)
MILP Model of iML1515_TFA_RBA
18928 variables, 16335 constraints, 153887 non-zero matrix coefficients
3750 enzymes, 6 process machines, 1899 TD reaction constraints
RBA enzyme efficiency constraints configured (C_EF_xxx, C_ER_xxx) ≤ 0
13 metabolite concentrations constrained
Genetic background, genes deleted: {'b3904', 'b3902', 'b0062', 'b0061', 'b3903', 'b0063', 'b0344'}
418 genes potentially active given flux distribution.


100%|████████████████████████████████████| 1613/1613 [1:35:55<00:00,  3.57s/it]
